# 🏃 Notebook 3: Threaded Motion Control & Safe Jogging

This notebook verifies physical or simulated hardware motion control by sending movement commands to axes that have completed the Servo On sequence. 

You will practice:
1. **Constant Velocity Control**: Moving the motor at a designated speed and stopping it.
2. **Hold-to-move Jogging**: A safety-critical jogging mechanism that automatically halts the motor if the control cycle is interrupted.
3. **Interactive HMI Control**: Operating the motor via user-friendly dashboard buttons and testing an **Emergency Stop (E-Stop)**.

*Prerequisite: You must execute `01_wmx_system_startup.ipynb` first to ensure that the designated axes are in the Servo On state.*

In [ ]:
import rclpy
from wmx_r2_message.msg import AxisVelocity
import ipywidgets as widgets
from IPython.display import display
import threading
import time
from wmx_utils import WmxClient

if not rclpy.ok():
    rclpy.init()

class WmxHmiController(WmxClient):
    def __init__(self):
        super().__init__(node_name='wmx_hmi_controller')
        
        # 불필요한 jog_pub, stop_cli 삭제 완료
        self.vel_pub = self.create_publisher(AxisVelocity, '/wmx/axis/velocity', 10)
        
        self.direction = 0
        self.is_running = False
        self.position = 0.0

        self.lbl_status = widgets.Label(value="Status: IDLE (Stopped)")
        self.lbl_pos = widgets.HTML(value="<h3>Current Position: 0.0 deg</h3>")

    def _loop(self):
        target_speed = 100.0 if self.direction > 0 else -100.0
        speeds = [target_speed] * len(self.axis_list)
        accs = [100.0] * len(self.axis_list)
        decs = [100.0] * len(self.axis_list)
                
        while self.is_running:
            # 50ms interval loop (20 FPS) for real-time browser feedback simulation
            self.publish_vel(self.axis_list, speeds, accs, decs)
            self.position += self.direction * 1.5
            self.lbl_pos.value = f"<h3>Current Position: <span style='color: #007acc;'>{self.position:.1f} deg</span></h3>"
            time.sleep(0.05)
            
        # Decelerate all controlled axes to a safe stop when STOP is clicked
        zero_speeds = [0.0] * len(self.axis_list)
        self.publish_vel(self.axis_list, zero_speeds, accs, decs)

    def publish_vel(self, index, velocity, acc, dec):
        msg = AxisVelocity()
        msg.index = index
        msg.velocity = velocity
        msg.acc = acc
        msg.dec = dec
        self.vel_pub.publish(msg)

    def start_move(self, direction):
        if not self.is_running:
            self.direction = direction
            self.is_running = True
            dir_text = "Rotating Positive (CW) 🔄" if direction > 0 else "Rotating Negative (CCW) 🔄"
            self.lbl_status.value = f"Status: {dir_text}"
            threading.Thread(target=self._loop, daemon=True).start()

    def stop_move(self):
        self.is_running = False
        self.direction = 0
        self.lbl_status.value = "Status: IDLE (Stopped)"

# Instantiate the controller
hmi = WmxHmiController()
print(f"✅ HMI Controller Ready. Controlling Axes: {hmi.axis_list}")

### 1. Interactive Dashboard (HMI) Control
* This control dashboard uses a multi-threaded Python approach combined with `ipywidgets`.
* Click **[▶ Positive (CW)]** or **[◀ Negative (CCW)]** to spin the actual motor. The thread updates the browser with simulated coordinate feedback in real-time.
* Click **[■ STOP]** to safely bring the physical motor to a halt.

In [ ]:
# Instantiate control dashboard buttons
btn_cw = widgets.Button(description="▶ Positive (CW)", button_style="success", layout=widgets.Layout(width="140px", height="40px"))
btn_stop = widgets.Button(description="■ STOP", button_style="danger", layout=widgets.Layout(width="90px", height="40px"))
btn_ccw = widgets.Button(description="◀ Negative (CCW)", button_style="warning", layout=widgets.Layout(width="140px", height="40px"))

# Bind buttons directly to controller state machine functions
btn_cw.on_click(lambda _: hmi.start_move(1))
btn_ccw.on_click(lambda _: hmi.start_move(-1))
btn_stop.on_click(lambda _: hmi.stop_move())

# Arrange the GUI layout nicely and display
controls = widgets.HBox([btn_ccw, btn_stop, btn_cw], layout=widgets.Layout(justify_content="center", margin="10px 0"))
display_box = widgets.VBox([hmi.lbl_status, hmi.lbl_pos], layout=widgets.Layout(align_items="center"))
panel = widgets.VBox([controls, display_box], layout=widgets.Layout(border="1px solid #ccc", padding="15px", width="420px", align_items="center"))

print("📬 Operate the motor and monitor the live coordinate stream below:")
display(panel)
